# S-SONDO Linear Probe on ESC-50

Demonstrates downstream finetuning with a **frozen S-SONDO backbone + linear classifier** on ESC-50 (50 environmental sound classes).

This is the standard evaluation protocol for audio embedding models:
1. Load pretrained model via `pip install ssondo` (auto-downloads from [HF Hub](https://huggingface.co/mohammedali2501/ssondo))
2. Freeze the backbone
3. Train a linear classifier on top
4. Report 5-fold cross-validation accuracy

In [ ]:
import os

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset, Audio
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

from ssondo import get_ssondo

SEED = 42
TARGET_SR = 32000
MODEL_NAME = "matpac-mobilenetv3"
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 64

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load S-SONDO Model

The model is auto-downloaded from [Hugging Face Hub](https://huggingface.co/mohammedali2501/ssondo) and the backbone is frozen for linear probing.

In [ ]:
model = get_ssondo(MODEL_NAME, device=DEVICE)
model.freeze_backbone()
print(f'Model: {MODEL_NAME}')
print(f'Embedding dim: {model.embedding_dim}')
print(f'Backbone frozen: {not next(model.backbone.parameters()).requires_grad}')

## 2. Load ESC-50 Dataset

In [ ]:
ds = load_dataset('ashraq/esc50', split='train')
ds = ds.cast_column('audio', Audio(sampling_rate=TARGET_SR))
n_classes = len(set(ds['target']))
print(f'ESC-50: {len(ds)} samples, {n_classes} classes')

for i in range(3):
    sample = ds[i]
    audio = sample['audio']
    print(f"  Sample {i}: category='{sample['category']}', duration={len(audio['array'])/audio['sampling_rate']:.1f}s")

## 3. Extract Embeddings (frozen backbone, one-time)

We extract all embeddings once and cache them in memory. Since the backbone is frozen, these won't change during training.

In [ ]:
all_emb = []
all_labels = []

for sample in tqdm(ds, desc='Extracting embeddings'):
    audio = sample['audio']
    waveform = torch.tensor(audio['array'], dtype=torch.float32)

    if waveform.ndim > 1:
        waveform = waveform.mean(dim=0)

    # Pad to 10s if shorter
    min_samples = TARGET_SR * 10
    if waveform.shape[0] < min_samples:
        waveform = torch.nn.functional.pad(waveform, (0, min_samples - waveform.shape[0]))

    waveform = waveform.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        emb = model.get_embeddings(waveform)

    all_emb.append(emb.squeeze(0).cpu())
    all_labels.append(sample['target'])

embeddings = torch.stack(all_emb)
labels = torch.tensor(all_labels, dtype=torch.long)
print(f'Embeddings: {embeddings.shape}')

## 4. Linear Probe (5-Fold Cross-Validation)

We follow ESC-50's predefined 5-fold split. For each fold, we train a linear classifier (only ~48K params) on the frozen embeddings.

In [ ]:
def train_linear_probe(embeddings, labels, train_idx, val_idx, emb_dim, n_classes, device):
    X_train = embeddings[train_idx].to(device)
    y_train = labels[train_idx].to(device)
    X_val = embeddings[val_idx].to(device)
    y_val = labels[val_idx].to(device)

    head = nn.Linear(emb_dim, n_classes).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    for epoch in range(EPOCHS):
        head.train()
        perm = torch.randperm(len(X_train))
        for i in range(0, len(X_train), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            logits = head(X_train[idx])
            loss = criterion(logits, y_train[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        head.eval()
        with torch.no_grad():
            val_preds = head(X_val).argmax(dim=1).cpu().numpy()
            val_acc = accuracy_score(y_val.cpu().numpy(), val_preds)
        if val_acc > best_acc:
            best_acc = val_acc

    return best_acc

In [ ]:
fold_ids = np.array(ds['fold'])
unique_folds = sorted(set(ds['fold']))

fold_accuracies = []
for fold in unique_folds:
    val_idx = np.where(fold_ids == fold)[0]
    train_idx = np.where(fold_ids != fold)[0]

    acc = train_linear_probe(
        embeddings, labels, train_idx, val_idx,
        model.embedding_dim, n_classes, DEVICE,
    )
    fold_accuracies.append(acc)
    print(f'Fold {fold}: {acc:.4f}')

mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies)
print(f'\nMean Accuracy: {mean_acc:.4f} +/- {std_acc:.4f}')

## 5. Results

In [ ]:
print('=' * 60)
print('S-SONDO LINEAR PROBE RESULTS')
print('=' * 60)
print(f'Model:    {MODEL_NAME} (auto-downloaded from HF Hub)')
print(f'Dataset:  ESC-50 ({len(ds)} samples, {n_classes} classes)')
print(f'Protocol: 5-fold CV, frozen backbone + linear head')
print(f'Backbone params: {sum(p.numel() for p in model.backbone.parameters()):,} (frozen)')
print(f'Head params:     {model.embedding_dim * n_classes + n_classes:,} (trainable)')
print(f'Epochs:   {EPOCHS}')
print()
for fold, acc in zip(unique_folds, fold_accuracies):
    print(f'  Fold {fold}: {acc:.4f}')
print(f'\n  Mean Accuracy: {mean_acc:.4f} +/- {std_acc:.4f}')
print('=' * 60)